## Khởi tạo dữ liệu

In [4]:
import pandas as pd

# Tạo dữ liệu từ bảng
data = {
    "Mã hóa đơn": ["o1", "o1", "o1", "o2", "o2", "o2", "o3", "o3", "o3", "o4", "o4", "o4", "o5", "o5"],
    "Mã hàng": ["i1", "i2", "i3", "i2", "i3", "i4", "i2", "i3", "i4", "i1", "i2", "i3", "i3", "i4"]
}

# Chuyển thành DataFrame
df = pd.DataFrame(data)

# Xuất ra file CSV
csv_path = 'hoa_don_ma_hang.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

df

,Mã hóa đơn,Mã hàng
0,o1,i1
1,o1,i2
2,o1,i3
3,o2,i2
4,o2,i3
5,o2,i4
6,o3,i2
7,o3,i3
8,o3,i4
9,o4,i1


## Tìm tập phổ biến thỏa ngưỡng từ ngưỡng phổ biến được người dùng nhập vào

In [5]:
# Nhập min-support
while True:
    try:
        min_support = float(input("Nhập min-support (0 < min-support < 1): "))
        if 0 < min_support < 1:
            break
        else:
            print("Giá trị min-support phải nằm trong khoảng (0, 1). Vui lòng thử lại.")
    except ValueError:
        print("Giá trị nhập vào không hợp lệ. Vui lòng nhập một số thực.")
        
# Gom transaction cho bộ dữ liệu
transactions = df.groupby('Mã hóa đơn')['Mã hàng'].apply(list).tolist()
num_transactions = len(transactions)
print(transactions)

# Hàm tính support
def calculate_support(itemset, transactions):
    count = 0
    for trans in transactions:
        if set(itemset).issubset(set(trans)):
            count += 1
    return count / num_transactions

# Thực hiện giải thuật Apriori
def apriori(transactions, min_support):
    # Tập tất cả các item duy nhất
    all_items = sorted(set(item for trans in transactions for item in trans))
    k = 1
    frequent_itemsets = []
    
    while True:
        # Sinh candidate k-itemset
        if k == 1:
            candidate_itemsets = [[item] for item in all_items]
        else:
            # Sinh từ (k-1)-itemset
            candidate_itemsets = [
                sorted(list(set(itemset1).union(itemset2)))
                for i, itemset1 in enumerate(frequent_itemsets_last)
                for itemset2 in frequent_itemsets_last[i+1:]
                if len(set(itemset1).union(itemset2)) == k
            ]
            
        # Lọc tập ứng viên thỏa min_support
        candidate_itemsets = [
            itemset for itemset in candidate_itemsets
            if calculate_support(itemset, transactions) >= min_support
        ]
        
        # Lưu vào frequent_itemsets
        frequent_itemsets.extend(candidate_itemsets)
        
        # Nếu không còn tập ứng viên, dừng
        if not candidate_itemsets:
            break
        
        # Chuẩn bị cho vòng lặp tiếp theo
        frequent_itemsets_last = candidate_itemsets
        k += 1
    
    # Loại các tập trùng và sắp xếp lại theo thứ tự từ 1-set đến n-set
    frequent_itemsets = [list(itemset) for itemset in set([tuple(itemset) for itemset in frequent_itemsets])]
    frequent_itemsets = sorted(frequent_itemsets, key=lambda x: (len(x), x))
    
    # Xuất tất cả các tập phổ biến thỏa min-supp
    return frequent_itemsets

# Tìm tập phổ biến
frequent_itemsets = apriori(transactions, min_support)
print('Tất cả các tập phổ biến thỏa min-support:')
for itemset in frequent_itemsets:
    print(itemset)

[['i1', 'i2', 'i3'], ['i2', 'i3', 'i4'], ['i2', 'i3', 'i4'], ['i1', 'i2', 'i3'], ['i3', 'i4']]
Tất cả các tập phổ biến thỏa min-support:
['i1']
['i2']
['i3']
['i4']
['i1', 'i2']
['i1', 'i3']
['i2', 'i3']
['i2', 'i4']
['i3', 'i4']
['i1', 'i2', 'i3']
['i2', 'i3', 'i4']


## Tìm các tập phổ biến tối đại dựa theo các tập phổ biến thỏa min-support đã tìm

In [6]:
frequent_itemsets
# Tìm tập phổ biến tối đại theo danh sách đã tìm được
def find_maximal(frequent_itemsets):
    maximal_itemsets = []
    for itemset in frequent_itemsets:
        maximal = True
        for other_itemset in frequent_itemsets:
            if set(itemset) < set(other_itemset):  # Kiểm tra X ⊂ Y
                maximal = False
                break
        if maximal:
            maximal_itemsets.append(itemset)
    return maximal_itemsets

# Xuất ra danh sách các tập phổ biến tối đại
maximal_itemsets = find_maximal(frequent_itemsets)
print('Tất cả các tập phổ biến tối đại:')
for itemset in maximal_itemsets:
    print(itemset)

Tất cả các tập phổ biến tối đại:
['i1', 'i2', 'i3']
['i2', 'i3', 'i4']


## Rút ra các luật kết hợp từ tập phổ biến tối đại

In [7]:
from itertools import combinations

maximal_itemsets
# Tìm các luật kết hợp từ maximal_itemsets
def find_rules(maximal_itemsets):
    rules = []
    for itemset in maximal_itemsets:
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    antecedent = list(antecedent)
                    consequent = sorted(list(set(itemset) - set(antecedent)))
                    rules.append((antecedent, consequent))
    return rules

# Tìm tất cả các luật
rules = find_rules(maximal_itemsets)
print('Tất cả các luật:')
for rule in rules:
    print(f"{rule[0]} -> {rule[1]}")

Tất cả các luật:
['i1'] -> ['i2', 'i3']
['i2'] -> ['i1', 'i3']
['i3'] -> ['i1', 'i2']
['i1', 'i2'] -> ['i3']
['i1', 'i3'] -> ['i2']
['i2', 'i3'] -> ['i1']
['i2'] -> ['i3', 'i4']
['i3'] -> ['i2', 'i4']
['i4'] -> ['i2', 'i3']
['i2', 'i3'] -> ['i4']
['i2', 'i4'] -> ['i3']
['i3', 'i4'] -> ['i2']


## Từ danh sách các luật, chọn những luật thỏa độ tin cậy

In [8]:
# Nhập min-confidence
while True:
    try:
        min_confidence = float(input("Nhập min-confidence (0 < min-confidence < 1): "))
        if 0 < min_confidence < 1:
            break
        else:
            print("Giá trị min-confidence phải nằm trong khoảng (0, 1). Vui lòng thử lại.")
    except ValueError:
        print("Giá trị nhập vào không hợp lệ. Vui lòng nhập một số thực.")

# Hàm tính confidence
def calculate_confidence(antecedent, consequent, transactions):
    return calculate_support(antecedent + consequent, transactions) / calculate_support(antecedent, transactions)

# Tìm các luật thỏa min-confidence
def find_satisfied_rules(rules, transactions, min_confidence):
    satisfied_rules = []
    for rule in rules:
        if calculate_confidence(rule[0], rule[1], transactions) >= min_confidence:
            satisfied_rules.append(rule)
    return satisfied_rules

# Xuất ra danh sách các luật thỏa min-confidence
satisfied_rules = find_satisfied_rules(rules, transactions, min_confidence)
print('Tất cả các luật thỏa min-confidence:')
for rule in satisfied_rules:
    print(f"{rule[0]} -> {rule[1]}")

Tất cả các luật thỏa min-confidence:
['i1'] -> ['i2', 'i3']
['i1', 'i2'] -> ['i3']
['i1', 'i3'] -> ['i2']
['i4'] -> ['i2', 'i3']
['i2', 'i4'] -> ['i3']
['i3', 'i4'] -> ['i2']


## Xây dựng giao diện cho ứng dụng thực hiện tìm tập phổi biến và luật kết hợp (trong file 'Ung_dung.py' đính kèm)